# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a dataset defined by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset schema is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This sets up our environment and loads the dataset schema and data.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Examine the structure of the dataset, its record sets, and available fields. All entities are referenced by their `@id` attributes for unambiguous field selection.

In [ ]:
# List all record sets available in the dataset
print('Record Sets:')
record_sets = dataset.record_sets
for rs in record_sets:
    print(f" - @id: {rs['@id']} | name: {rs.get('name', '(no name)')}")

# For each record set, list fields with their @id and labels
for rs in record_sets:
    print(f"\nRecord Set: {rs.get('name', '(no name)')} (@id: {rs['@id']})")
    print('Fields:')
    fields = rs.get('field', [])
    # Ensure fields is always a list
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        if isinstance(f, dict):
            field_id = f['@id']
            field_name = f.get('name', '(no name)')
        else:
            # Some croissant schemas use just @id strings
            field_id = f
            field_name = ''
        print(f"   - @id: {field_id} | name: {field_name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis, referencing record set and field `@id`s from the previous overview.

In [ ]:
# Example: Select the main record set for survey responses
# List all record set @id's for reference
record_set_ids = [rs['@id'] for rs in record_sets]
print('Available record set @ids:', record_set_ids)

# For this dataset (as of 2024-06), there is typically only one main record set
main_record_set_id = record_set_ids[0]
print(f"Using main record set: {main_record_set_id}")

# Load all records from the main record set
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print('Columns (fields as @id):', df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's apply a few exploratory steps: filtering, normalizing, and grouping records. All fields are referenced using their `@id` value.

Suppose we want to analyze the PHQ-9 total score (commonly used for depression symptoms), filter respondents with high scores, normalize, and group by a categorical field such as gender.

In [ ]:
# Pick candidate field @id's (based on field listing in section 2). Update to actual @id for your exploration.
# We'll use '@id' examples -- in real usage, pick from the earlier output.
phq9_total_id = None
gender_id = None
for c in df.columns:
    cn = c.lower()
    # Try to guess sensible columns
    if 'phq9' in cn or 'phq_9' in cn or 'phq-9' in cn or ('total' in cn and 'phq' in cn):
        phq9_total_id = c
    if 'gender' in cn:
        gender_id = c
print('Attempting to use PHQ-9 total field:', phq9_total_id)
print('Attempting to use gender field:', gender_id)

# If not found, fallback to manual selection
if phq9_total_id is None or gender_id is None:
    print("Couldn't find candidate field(s) by name, please replace '@id's below based on your dataset.")
# Example fallback:
# phq9_total_id = '@id:phq9_total_score' (replace with actual @id)
# gender_id = '@id:gender' (replace with actual @id)

# EDA: Filter for participants with PHQ-9 > 10
eda_field = phq9_total_id if phq9_total_id is not None else df.columns[0]  # fallback
threshold = 10
filtered_df = df.copy()
if pd.api.types.is_numeric_dtype(filtered_df[eda_field]):
    filtered_df = filtered_df[filtered_df[eda_field] > threshold]
    print(f"Filtered records with {eda_field} > {threshold}:")
    print(filtered_df.head())

    norm_col = f"{eda_field}_normalized"
    filtered_df[norm_col] = (filtered_df[eda_field] - filtered_df[eda_field].mean()) / filtered_df[eda_field].std()
    print(f"Normalized {eda_field} for filtered records:")
    print(filtered_df[[eda_field, norm_col]].head())

    group_by = gender_id if gender_id is not None else None
    if group_by is not None and group_by in filtered_df.columns:
        grouped = filtered_df.groupby(group_by)[eda_field].mean()
        print(f"\nGrouped mean {eda_field} by {group_by}:")
        print(grouped)
else:
    print(f"Selected field {eda_field} is not numeric. Please check field selection.")

## 5. Visualization
Let's visualize the distribution of PHQ-9 (or whichever numeric field was selected) and its relationship to another categorical variable, such as gender.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot PHQ-9 score distribution (or appropriate field)
if pd.api.types.is_numeric_dtype(df[eda_field]):
    plt.figure(figsize=(8, 4))
    sns.histplot(df[eda_field].dropna(), kde=True)
    plt.title(f'Distribution of {eda_field}')
    plt.xlabel(eda_field)
    plt.show()

    if gender_id and gender_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[gender_id], y=df[eda_field])
        plt.title(f'{eda_field} by {gender_id}')
        plt.xlabel(gender_id)
        plt.ylabel(eda_field)
        plt.show()
else:
    print(f"Selected field {eda_field} is not numeric. Visualization skipped.")

## 6. Conclusion
We have demonstrated loading a FAIR²-compliant mental health dataset using Croissant and `mlcroissant`, explored its structure and fields using `@id` references, filtered and normalized PHQ-9 depression screening scores, and visualized their distribution, grouped by gender.

You can extend this notebook to other record sets, fields, or apply further analyses and modeling depending on your research questions.